# First Title
## Second Title
- Item 1
- Item 2
- Item 3 
**abcde**

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [9]:
df=pd.read_csv("RFM_raw data.csv") 
df.head

<bound method NDFrame.head of           Date     InvoNo CustomerKey     Amount
0     20180402  AP1803827       A1067   836.0000
1     20180403  AP1808235       A1129   836.0000
2     20180404  AP1807305       A1133   836.0000
3     20180405  AP1806145       A0464   836.0000
4     20180405  AP1803092       A0383   836.0000
...        ...        ...         ...        ...
9376  20190102  JA1905084       A0922  2472.7020
9377  20190102  JA1908815       A0738  3500.7984
9378  20190104  JA1903532       A0632  2365.4400
9379  20190104  JA1902149       A1036  6020.8400
9380  20190104  JA1907784       A1057   205.7000

[9381 rows x 4 columns]>

In [10]:
import pandas as pd

# -------------------------------------------------------------
# 步驟 1：重新讀取原始檔案（請換成你的真實路徑或檔案名稱）
# -------------------------------------------------------------

df = pd.read_csv("RFM_raw data.csv")       # 如果是 csv

# -------------------------------------------------------------
# 步驟 2：正確解析 YYYYMMDD 純數字日期
# -------------------------------------------------------------
# 關鍵：先轉成純整數字串（去掉可能的小數點 .0），再用 %Y%m%d 解析
df['Date'] = df['Date'].astype(str).str.split('.').str[0].str.strip()
df['Date'] = pd.to_datetime(df['Date'], format='%Y%m%d')

# 檢查一下全表最大日期，此時「必須」顯示 2019-01-04
max_date = df['Date'].max()
print("【正確解析後的全表最大日期】:", max_date)

# -------------------------------------------------------------
# 步驟 3：清理客戶編號格式（去掉空白）
# -------------------------------------------------------------
df['CustomerKey'] = df['CustomerKey'].astype(str).str.strip()

# -------------------------------------------------------------
# 步驟 4：分組計算 RFM
# -------------------------------------------------------------
result = df.groupby('CustomerKey').agg(
    F=('InvoNo', 'nunique'),
    M=('Amount', 'sum'),
    last_date=('Date', 'max')
).reset_index()

# 計算 R（天數差）
result['R'] = (max_date - result['last_date']).dt.days

# 欄位重新命名與排序
result = result.rename(columns={'CustomerKey': 'customer'})
result = result[['customer', 'F', 'M', 'R']]

# -------------------------------------------------------------
# 步驟 5：驗證 A0350 的結果
# -------------------------------------------------------------
print("\n【A0350 的正確結果】:")
print(result)

    customer   F            M  R
0      A0350  25  51238.11440  0
1      A0351   6  17890.31200  0
2      A0352  14  46424.03852  0
3      A0353  14  27607.68636  0
4      A0354   6   9973.92064  0
..       ...  ..          ... ..
727    A1413  23  52738.79456  0
728    A1415  20  53019.49664  0
729    A1416  24  49938.38378  0
730    A1419  22  64257.84520  0
731    A1420  14  25054.18342  0

[732 rows x 4 columns]
